> **참고용 사본 (reference copy).** 원본 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 사본입니다. 비공개 원천 데이터가 필요해 이 저장소만으로는 실행되지 않으며, 코드 구조와 분석 절차를 보여주는 용도입니다.

*Reference copy of a private notebook: outputs and paths removed; requires non-public data to run.*


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
SVI = pd.read_csv("./input/SVI.csv", index_col=0, parse_dates=True)
SVI = SVI.resample('ME').last()
SVI = np.log(SVI)
SVI.tail()

In [ ]:
# 로그 차분
ln_SVI = SVI.diff()
ln_SVI.tail()

In [ ]:
import statsmodels.api as sm

month_dummies = pd.get_dummies(SVI.index.month, prefix='M', drop_first=True)
month_dummies.index = SVI.index
X = sm.add_constant(month_dummies).astype(float)

SVI_resid = pd.DataFrame(index=SVI.index, columns=SVI.columns, dtype=float)

for col in SVI.columns:
    y = SVI[col].astype(float)
    mask = y.notna()
    if mask.sum() > X.shape[1]:
        SVI_resid.loc[mask, col] = sm.OLS(y.loc[mask], X.loc[mask]).fit().resid

SVI_resid.tail()

In [ ]:
import statsmodels.api as sm

# 월 정보
month = ln_SVI.index.month

# 월 더미 (1월 기준)
month_dummies = pd.get_dummies(month, prefix='M', drop_first=True)
month_dummies.index = ln_SVI.index

# 디자인 매트릭스
X = sm.add_constant(month_dummies).astype(float)

In [ ]:
# 컬럼별로 회귀
ASVI = pd.DataFrame(index=ln_SVI.index, columns=ln_SVI.columns, dtype=float)

for col in ln_SVI.columns:
    y = ln_SVI[col].astype(float)

    # 결측 제거
    mask = y.notna()

    if mask.sum() <= X.shape[1]:
        continue  # 표본 부족하면 스킵

    model = sm.OLS(y.loc[mask], X.loc[mask]).fit()

    # 계절성 제거된 ASVI = 잔차
    ASVI.loc[mask, col] = model.resid

ASVI.tail()

In [ ]:
W = 60
ASVI_oos = pd.DataFrame(index=ln_SVI.index, columns=ln_SVI.columns, dtype=float)

for col in ln_SVI.columns:
    y = ln_SVI[col].astype(float)

    for t in range(W, len(y)):
        train_idx = y.index[t-W:t]   # 직전 W(=60)개월
        test_idx  = y.index[t]       # 현재 시점 t

        train = pd.concat([y.loc[train_idx].rename('y'), X.loc[train_idx]], axis=1).dropna()
        test_x = X.loc[[test_idx]].dropna()

        if len(train) <= X.shape[1] or len(test_x) == 0:
            continue

        model = sm.OLS(train['y'], train.drop(columns='y')).fit()
        y_hat = model.predict(test_x).iloc[0]

        ASVI_oos.loc[test_idx, col] = y.loc[test_idx] - y_hat

ASVI_oos.tail()

In [ ]:
# 가져올 컬럼
cols = ['N_배당', 'N_배당금', 'N_배당_주제_1']

# 각 데이터프레임에서 필요한 컬럼만 선택 후 컬럼명 변경
svi_selected = SVI_resid[cols].rename(columns={
    'N_배당': 'SVI_배당',
    'N_배당금': 'SVI_배당금',
    'N_배당_주제_1': 'SVI_배당_배당금'
})


ln_svi_selected = ln_SVI[cols].rename(columns={
    'N_배당': 'ln_SVI_배당',
    'N_배당금': 'ln_SVI_배당금',
    'N_배당_주제_1': 'ln_SVI_배당_배당금'
})

asvi_selected = ASVI[cols].rename(columns={
    'N_배당': 'ASVI_배당',
    'N_배당금': 'ASVI_배당금',
    'N_배당_주제_1': 'ASVI_배당_배당금'
})

asvi_oos_selected = ASVI_oos[cols].rename(columns={
    'N_배당': 'ASVI_oos_배당',
    'N_배당금': 'ASVI_oos_배당금',
    'N_배당_주제_1': 'ASVI_oos_배당_배당금'
})

# 옆으로 합치기
svi_df = pd.concat([svi_selected, ln_svi_selected, asvi_selected, asvi_oos_selected], axis=1)

# 확인
svi_df.tail()

In [ ]:
svi_df.to_csv("./output/svi_processed.csv")